# Notebook 02 — MAPIE baseline

Roda o MAPIE para os 3 desfechos como baseline de conformal prediction. Vamos
comparar dois métodos:

- **LAC** (Least Ambiguous Classifier): o split conformal clássico. Score simples,
  pode gerar conjuntos vazios.
- **APS** (Adaptive Prediction Sets): score adaptativo, sem conjuntos vazios,
  geralmente com cobertura mais próxima do nominal.

Esses resultados são o ponto de referência contra o qual vamos comparar o Locart
(que vem nos próximos notebooks).

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q mapie

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.4/265.4 kB 5.6 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pickle
from sklearn.linear_model import LogisticRegression
from mapie.classification import SplitConformalClassifier

DRIVE = Path('/content/drive/MyDrive/projeto_violencia_mulher')
SAIDA = DRIVE / 'conformal_violence'

RANDOM_STATE = 42
ALPHA = 0.1                          # tolerância de erro
CONFIDENCE = 1 - ALPHA               # 0.9 = cobertura nominal de 90%

## 2. Carregando os dados do notebook 01

In [ ]:
with open(SAIDA / 'dados_conformal.pkl', 'rb') as f:
    dados = pickle.load(f)

X_train = dados['X_train']
X_cal = dados['X_cal']
X_test = dados['X_test']
y_train = dados['y_train']
y_cal = dados['y_cal']
y_test = dados['y_test']
feature_names = dados['feature_names']
ALVOS = dados['alvos']

print(f'Treino: {X_train.shape[0]:,} x {X_train.shape[1]}')
print(f'Calibração: {X_cal.shape[0]:,}')
print(f'Teste: {X_test.shape[0]:,}')

Treino: 20,000 x 275
Calibração: 743,021
Teste: 852,044


## 3. Treinando uma logística para cada desfecho

logística L1

In [ ]:
modelos = {}
for alvo in ALVOS:
    modelo = LogisticRegression(
        penalty='l1', solver='liblinear', C=1.0,
        max_iter=500, random_state=RANDOM_STATE
    )
    modelo.fit(X_train, y_train[alvo].values)
    modelos[alvo] = modelo

## 4. Funções auxiliares para avaliar conjuntos de predição

A API nova do MAPIE mudou os nomes das métricas de avaliação, então calculo
manualmente. Cobertura = quantos casos tem a classe verdadeira no conjunto.
Tamanho médio = média de classes por conjunto.

In [ ]:
def avaliar_conjuntos(y_verdadeiro, conjuntos):
    """Calcula cobertura, tamanho médio e distribuição de tamanhos."""
    n = len(y_verdadeiro)
    # cobertura: 1 se a classe verdadeira está no conjunto
    cobertos = np.array([conjuntos[i, y_verdadeiro[i]] for i in range(n)])
    cobertura = cobertos.mean()

    tamanhos = conjuntos.sum(axis=1)
    return {
        'cobertura': float(cobertura),
        'tamanho_medio': float(tamanhos.mean()),
        'pct_vazios': float((tamanhos == 0).mean()),
        'pct_unitarios': float((tamanhos == 1).mean()),
        'pct_duplos': float((tamanhos == 2).mean()),
    }

## 5. MAPIE com método LAC

LAC é o método mais direto: score = 1 - p(classe verdadeira), quantil global,
conjunto = classes com score abaixo do quantil. Quando o modelo é muito confiante,
pode gerar conjuntos vazios.

In [ ]:
resultados_lac = {}

for alvo in ALVOS:
    modelo = modelos[alvo]
    y_cal_alvo = y_cal[alvo].values
    y_test_alvo = y_test[alvo].values

    mapie = SplitConformalClassifier(
        estimator=modelo,
        confidence_level=CONFIDENCE,
        conformity_score='lac',
        prefit=True,
        random_state=RANDOM_STATE,
    )
    mapie.conformalize(X_cal, y_cal_alvo)

    _, conjuntos = mapie.predict_set(X_test)
    conjuntos = conjuntos[:, :, 0]   # (n_test, n_classes)

    res = avaliar_conjuntos(y_test_alvo, conjuntos)
    res['conjuntos'] = conjuntos
    resultados_lac[alvo] = res

    print(f"{alvo}: cob={res['cobertura']:.4f}, tam={res['tamanho_medio']:.4f}, vazios={res['pct_vazios']:.1%}")

y_fisic: cob=0.9064, tam=1.1243, vazios=0.0%
y_psico: cob=0.9063, tam=1.1591, vazios=0.0%
y_sexu: cob=0.8946, tam=0.9523, vazios=4.8%


## 6. MAPIE com método APS

APS (Adaptive Prediction Sets) é mais sofisticado. Em vez de comparar
`1 - p(verdadeira)` contra um threshold, soma probabilidades até atingir o
threshold de cobertura. Resultado: nunca gera conjuntos vazios e tem cobertura
mais próxima do nominal.

In [ ]:
resultados_aps = {}

for alvo in ALVOS:
    modelo = modelos[alvo]
    y_cal_alvo = y_cal[alvo].values
    y_test_alvo = y_test[alvo].values

    mapie = SplitConformalClassifier(
    estimator=modelo,
    confidence_level=CONFIDENCE,
    conformity_score='lac',
    prefit=True,
    random_state=RANDOM_STATE,)
    mapie.conformalize(X_cal, y_cal_alvo)

    _, conjuntos = mapie.predict_set(
        X_test,
        conformity_score_params={'include_last_label': True}
    )
    conjuntos = conjuntos[:, :, 0]

    res = avaliar_conjuntos(y_test_alvo, conjuntos)
    res['conjuntos'] = conjuntos
    resultados_aps[alvo] = res

    print(f"{alvo}: cob={res['cobertura']:.4f}, tam={res['tamanho_medio']:.4f}, vazios={res['pct_vazios']:.1%}")

y_fisic: cob=0.9064, tam=1.1243, vazios=0.0%
y_psico: cob=0.9063, tam=1.1591, vazios=0.0%
y_sexu: cob=0.8946, tam=0.9523, vazios=4.8%


## 7. Comparação resumida

Tabela comparando LAC e APS para os 3 desfechos. A cobertura deve estar próxima
de 90% em ambos. APS geralmente tem cobertura ligeiramente melhor e zero conjuntos
vazios. LAC pode gerar mais unitários (mais informativo) mas também mais vazios.

In [ ]:
linhas = []
for alvo in ALVOS:
    linhas.append({
        'desfecho': alvo, 'metodo': 'LAC',
        **{k: v for k, v in resultados_lac[alvo].items() if k != 'conjuntos'},
    })
    linhas.append({
        'desfecho': alvo, 'metodo': 'APS',
        **{k: v for k, v in resultados_aps[alvo].items() if k != 'conjuntos'},
    })

tabela = pd.DataFrame(linhas)
print(tabela.to_string(index=False))

desfecho metodo  cobertura  tamanho_medio  pct_vazios  pct_unitarios  pct_duplos
 y_fisic    LAC   0.906377       1.124264     0.00000       0.875736    0.124264
 y_fisic    APS   0.906377       1.124264     0.00000       0.875736    0.124264
 y_psico    LAC   0.906337       1.159075     0.00000       0.840925    0.159075
 y_psico    APS   0.906337       1.159075     0.00000       0.840925    0.159075
  y_sexu    LAC   0.894597       0.952250     0.04775       0.952250    0.000000
  y_sexu    APS   0.894597       0.952250     0.04775       0.952250    0.000000


## 8. Salvando os modelos e os resultados

In [ ]:
resultados_baseline = {
    'modelos': modelos,
    'lac': resultados_lac,
    'aps': resultados_aps,
    'alpha': ALPHA,
    'confidence': CONFIDENCE,
    'tabela_comparativa': tabela,
}

with open(SAIDA / 'resultados_baseline.pkl', 'wb') as f:
    pickle.dump(resultados_baseline, f)

print(f'Salvo em {SAIDA / "resultados_baseline.pkl"}')

Salvo em /content/drive/MyDrive/projeto_violencia_mulher/conformal_violence/resultados_baseline.pkl
